# MCMC Uncertainty Estimation

This notebook adds Markov-chain Monte Carlo (MCMC) uncertainty estimation on
top of the global 2D fit from [`01_basic_fitting`](../01_basic_fitting/example.ipynb).
See that notebook for the fit itself; here we focus on sampling the posterior
of the fitted parameters with `lmfit.emcee`.

The first cell re-runs notebook 01 in this kernel via the `%run` magic, so its
fitted `file` / `project` are available below — the same preamble pattern that
`11_save_load_export` uses to inherit `10_model_comparison`.

In [ ]:
%%capture
# %cd into notebook 01's dir so its project.yaml + data/models resolve;
# %run executes that notebook in the current kernel (variables stay live);
# %cd - returns here. %%capture swallows notebook 01's plots/output — its
# show_output: 1 stays correct for direct interactive use.
# (The path is quoted because a bare ../01_... trips IPython's tokenizer:
# 01 reads as an invalid leading-zero numeric literal.)
%cd -q "../01_basic_fitting"
%run example.ipynb
%cd -q -

In [ ]:
# Heartbeat: confirm notebook 01's pipeline landed in this kernel.
n_free = sum(p.vary for p in file.model_2d.lmfit_pars.values())
print(f"file:        {file.name}")
print(f"2D model:    {file.model_2d.name}")
print(f"free params: {n_free}")

## 1. Sample the Posterior with MCMC

`MC(use_mc=1, ...)` switches on `lmfit.emcee`. We re-run the 2D fit starting
from notebook 01's least-squares solution, then sample the posterior around it.

Because the inherited project has `show_output: 1` and `auto_export: False`,
the **corner plot** (pairwise marginal posteriors) and the **walker
acceptance-ratio plot** are drawn inline and nothing is written to disk.

In [ ]:
from trspecfit.utils.lmfit import MC

# nwalkers must be >= 2 * (number of varying parameters); emcee also adds a
# __lnsigma nuisance parameter when is_weighted=False. steps/burn are kept
# small here for a fast demo — see the tips at the end for production values.
mc = MC(use_mc=1, nwalkers=20, steps=500, burn=100, thin=1)

# try_ci=0: skip the linear confidence interval and let MCMC do the uncertainty
# work. stages=1 is enough since we start from notebook 01's converged fit.
file.fit_2d(model_name='2D', stages=1, try_ci=0, mc_settings=mc)

## 2. Read Off the MCMC Confidence Intervals

`fit_2d` stores the fit result as `file.model_2d.result =
[par_ini, par_fin, conf_ci, emcee_fin, emcee_ci]`. The MCMC outputs are the
last two: `emcee_fin` (the raw `lmfit` MCMC result, with `.flatchain` and
`.acceptance_fraction`) and `emcee_ci` (a tidy table of posterior quantiles).

The `emcee_ci` columns give the best fit (posterior median) with ±1σ/2σ/3σ
bands from the flatchain quantiles — one row per sampled parameter. Compare
these against the least-squares `stderr` from
`file.get_fit_results(fit_type="2d")` shown in notebook 01: where the posterior
is Gaussian and uncorrelated the two agree; where it is not, the corner plot
above and these intervals tell the fuller story.

In [ ]:
emcee_fin = file.model_2d.result[3]
print(
    f"mean acceptance fraction: {emcee_fin.acceptance_fraction.mean():.2f}"
    "  (aim for 0.2–0.5)"
)

emcee_ci = file.model_2d.result[4]
emcee_ci

## Tips for MCMC Uncertainty Estimation

- **When to use it.** `try_ci=1` gives a fast linear (Gaussian) confidence
  interval. Reach for MCMC when parameters are correlated or the posterior is
  non-Gaussian — the corner plot reveals both.
- **Acceptance fraction** should sit around **0.2–0.5**. Much lower means the
  walkers are stuck (try more steps or a different start); much higher means
  the steps are too small.
- **Burn-in / thinning.** `burn` discards the first steps while walkers find
  the posterior; `thin` keeps every n-th sample to reduce autocorrelation.
- **nwalkers** must be at least `2 × n_varying` (emcee adds a `__lnsigma`
  nuisance parameter when `is_weighted=False`).
- **This notebook keeps `steps` small for speed.** For publication-quality
  posteriors use `steps=5000+`, `burn=500`, `thin=5`.

See [`01_basic_fitting`](../01_basic_fitting/example.ipynb) for the underlying
fit and [`11_save_load_export`](../11_save_load_export/example.ipynb) to persist
results — the σ snapshot saved with each fit travels with the archive.